In [ ]:
import json
import re

def extract_answer_universal(text):
    """
    增强版统一答案提取函数。
    """
    if not text:
        return "FAILED"

    # 1. 优先匹配 LaTeX 中的 \boxed{...} 格式
    boxed_pattern = r'\\boxed\{\s*([A-D])(?:\.|\s|})'
    boxed_matches = re.findall(boxed_pattern, text, re.IGNORECASE)
    if boxed_matches:
        return boxed_matches[-1].upper()

    # 2. 增强型综合匹配模式
    patterns = [
        r'(?:answer|final answer|correct answer is):?\s*\**\s*\(?([A-D])\)?',
        r'\\boxed\{([A-D])\}' 
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].upper()

    # 3. 兜底策略
    fallback_pattern = r'(?:^|\s)\**([A-D])\**(?:\.|,|$|\s)'
    fallback_matches = re.findall(fallback_pattern, text, re.IGNORECASE)
    if fallback_matches:
        return fallback_matches[-1].upper()

    return "FAILED"

def calculate_accuracy(file_path):
    total_count = 0
    original_correct_count = 0
    regex_correct_count = 0
    total_token_length = 0  # 新增：用于累加 token 长度
    
    failed_extractions = []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                data = json.loads(line)
                total_count += 1
                
                # --- 1. 统计原始正确率 ---
                if data.get("correct") is True:
                    original_correct_count += 1
                
                # --- 2. 使用正则提取并统计正确率 ---
                gt = str(data.get("gt_answer", "")).strip().upper()
                full_output = data.get("full_output", "")
                
                pred_regex = extract_answer_universal(full_output)
                
                if pred_regex == gt:
                    regex_correct_count += 1
                elif pred_regex == "FAILED":
                    failed_extractions.append(data.get("id"))

                # --- 3. 累加 Token 长度 ---
                # 使用 .get(key, 0) 防止键不存在时报错
                total_token_length += data.get("token_length", 0)

        # 计算百分比和平均值
        original_acc = (original_correct_count / total_count) * 100 if total_count > 0 else 0
        regex_acc = (regex_correct_count / total_count) * 100 if total_count > 0 else 0
        avg_token_length = total_token_length / total_count if total_count > 0 else 0

        # --- 打印报告 ---
        print("-" * 40)
        print(f"📊 数据统计报告 (总计: {total_count} 条)")
        print("-" * 40)
        print(f"✅ 原始标注正确率: {original_acc:.2f}% ({original_correct_count}/{total_count})")
        print(f"🔍 正则提取正确率: {regex_acc:.2f}% ({regex_correct_count}/{total_count})")
        print(f"📏 平均 Token 长度: {avg_token_length:.2f}")
        print("-" * 40)
        
        if failed_extractions:
            print(f"⚠️ 提示: 有 {len(failed_extractions)} 条数据正则匹配失败。")
            
    except FileNotFoundError:
        print(f"❌ 错误: 找不到文件 {file_path}")
    except Exception as e:
        print(f"❌ 运行出错: {e}")

# --- 运行 ---
calculate_accuracy('./scienceqa/scienceqa_qwen2vl_ori_results.jsonl')
calculate_accuracy('./m3cot/m3cot_qwen2vl_ori_results.jsonl')